# ¿Qué solución conviene para mi proyecto?

Comparo mi mejor ensamblaje de la sesión pasada (el bosque aleatorio), una red neuronal simple y un LLM para predecir las unidades vendidas. La preparación y la división se importan de `pipeline_proyecto.py`.

In [1]:
import time

import pandas as pd
from sklearn.base import clone
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from pipeline_proyecto import X_ent, X_prueba, preparacion, y_ent, y_prueba

resultados = {}
entrenados = {}


def evaluar(nombre, modelo):
    pipeline = Pipeline([("preparar", clone(preparacion)), ("modelo", modelo)])
    inicio = time.perf_counter()
    pipeline.fit(X_ent, y_ent)
    segundos = time.perf_counter() - inicio
    mae_prueba = mean_absolute_error(y_prueba, pipeline.predict(X_prueba))
    resultados[nombre] = {"modelo": nombre, "mae_prueba": round(mae_prueba, 4), "segundos": round(segundos, 1)}
    entrenados[nombre] = pipeline
    print(f"{nombre:18s} prueba {mae_prueba:.4f}   {segundos:6.1f} s", flush=True)

Filas y columnas del archivo: (468820, 15)
Entrenamiento: fechas 2011-01-29 a 2015-12-31
Prueba: fechas 2016-01-01 a 2016-05-22
Vacios en entrenamiento: {}
Vacios en prueba: {'precio_relativo': 21}


Entrenamiento: (424975, 26) Prueba: (43845, 26)
['numericas__SELL_PRICE', 'numericas__precio_relativo', 'numericas__dias_desde_lanzamiento', 'categoricas__CAT_ID_FOODS', 'categoricas__CAT_ID_HOBBIES', 'categoricas__CAT_ID_HOUSEHOLD', 'categoricas__STORE_ID_CA_1', 'categoricas__STORE_ID_CA_2', 'categoricas__STORE_ID_CA_3', 'categoricas__STORE_ID_CA_4', 'categoricas__STORE_ID_TX_1', 'categoricas__STORE_ID_TX_2', 'categoricas__STORE_ID_TX_3', 'categoricas__STORE_ID_WI_1', 'categoricas__STORE_ID_WI_2', 'categoricas__STORE_ID_WI_3', 'categoricas__WEEKDAY_Friday', 'categoricas__WEEKDAY_Monday', 'categoricas__WEEKDAY_Saturday', 'categoricas__WEEKDAY_Sunday', 'categoricas__WEEKDAY_Thursday', 'categoricas__WEEKDAY_Tuesday', 'categoricas__WEEKDAY_Wednesday', 'ya_listas__IS_WEEKEND', 'ya_listas__IS_EVENT_DAY', 'ya_listas__IS_SNAP']
Medianas que aprendio el imputador: {'SELL_PRICE': 3.44, 'precio_relativo': 1.0, 'dias_desde_lanzamiento': 729.0}
Medianas de entrenamiento: {'SELL_PRICE': 3.44, 'prec

## Los dos modelos

In [2]:
evaluar("bosque aleatorio", RandomForestRegressor(n_estimators=100, max_depth=12, min_samples_leaf=50, n_jobs=-1, random_state=42))
evaluar("red neuronal", MLPRegressor(hidden_layer_sizes=(64,), max_iter=200, random_state=42))

modelo_red = entrenados["red neuronal"].named_steps["modelo"]
parametros_red = sum(matriz.size for matriz in modelo_red.coefs_) + sum(vector.size for vector in modelo_red.intercepts_)
resultados["red neuronal"]["parametros"] = parametros_red
print("Parametros de la red:", parametros_red)

tabla = pd.DataFrame(resultados.values())
tabla.to_csv("comparar_proyecto.csv", index=False)
print(tabla.to_string(index=False))

bosque aleatorio   prueba 1.5069     80.6 s


red neuronal       prueba 1.6706    216.9 s


Parametros de la red: 1793
          modelo  mae_prueba  segundos  parametros
bosque aleatorio      1.5069      80.6         NaN
    red neuronal      1.6706     216.9      1793.0


## La prueba del LLM

10 filas de prueba y 30 de entrenamiento como ejemplos. El prompt no incluye las unidades reales de las 10 filas. Las respuestas del LLM (Claude Opus 5.5) están en `respuestas_proyecto.txt`.

In [3]:
muestra = X_prueba.sample(10, random_state=7)
y_muestra = y_prueba.loc[muestra.index]
ejemplos = X_ent.sample(30, random_state=7)

filas_ejemplo = ejemplos.copy().round(4)
filas_ejemplo["UNITS"] = y_ent.loc[ejemplos.index].values
prompt = (
    "Estimas cuantas unidades de un producto se venden en una tienda Walmart en un dia. Abajo hay 30 filas con sus "
    "unidades reales (UNITS) y 10 filas sin ese valor. Responde solo con 10 numeros, uno por linea, en el mismo orden "
    "de las filas sin unidades, sin texto adicional. precio_relativo es el precio del dia dividido para el precio "
    "promedio del producto, y dias_desde_lanzamiento son los dias que el producto lleva a la venta.\n\n"
    "Filas con unidades:\n" + filas_ejemplo.to_csv(index=False)
    + "\nFilas sin unidades:\n" + muestra.round(4).to_csv(index=False)
)
with open("prompt_proyecto.txt", "w") as archivo:
    archivo.write(prompt)

with open("respuestas_proyecto.txt") as archivo:
    predicciones_llm = [float(linea) for linea in archivo if linea.strip()]

filas = {
    "real": y_muestra.to_numpy(),
    "LLM": predicciones_llm,
    "bosque aleatorio": entrenados["bosque aleatorio"].predict(muestra).round(2),
    "red neuronal": entrenados["red neuronal"].predict(muestra).round(2),
}
print(pd.DataFrame(filas).to_string(index=False))
print()
for nombre in ["LLM", "bosque aleatorio", "red neuronal"]:
    print(f"MAE en las 10 filas, {nombre:18s} {mean_absolute_error(y_muestra, filas[nombre]):.4f}")
print(f"MAE en las 10 filas, siempre 0         {y_muestra.abs().mean():.4f}")

 real  LLM  bosque aleatorio  red neuronal
    1  1.0              0.75          0.28
    0  0.0              0.40          0.54
    0  1.0              0.35          0.51
    0  2.0              1.12          1.30
    0  0.0              1.11          1.11
    0  1.0              2.22          1.63
    0  1.0              3.31          4.66
   18  0.0              0.81          0.82
    0  1.0              1.25         -0.80
    0  3.0              2.46          2.24

MAE en las 10 filas, LLM                2.7000
MAE en las 10 filas, bosque aleatorio   2.9660
MAE en las 10 filas, red neuronal       3.0690
MAE en las 10 filas, siempre 0         1.9000


## El costo del LLM

Hay 3049 productos en 10 tiendas: 30,490 predicciones por día y, con 10 filas por prompt, 3049 prompts. Uso el mismo precio del Ejercicio 2 (4 dólares por millón de tokens de entrada), un token cada 4 caracteres, y no cuento los tokens de salida.

In [4]:
PRECIO_POR_MILLON_DE_TOKENS = 4.00
tokens_por_prompt = len(prompt) / 4
prompts_al_mes = 3049 * 30
costo_mes = tokens_por_prompt * prompts_al_mes * PRECIO_POR_MILLON_DE_TOKENS / 1_000_000
print(f"Prompt: {len(prompt)} caracteres, {tokens_por_prompt:.0f} tokens | {prompts_al_mes} prompts al mes | costo: {costo_mes:.2f} dolares al mes")

Prompt: 2452 caracteres, 613 tokens | 91470 prompts al mes | costo: 224.28 dolares al mes
